In [1]:
from google.colab import files

uploaded = files.upload()

Saving table6_2026_04.csv to table6_2026_04.csv


In [ ]:
import pandas as pd

df = pd.read_csv("table6_2026_04.csv")

df.head()

In [4]:
df.shape

(1981, 18)

In [ ]:
df["project_code"].nunique()

In [ ]:
df.head(12)

In [ ]:
df.info()

In [10]:
df["project_code"].nunique()

1981

In [ ]:
# Convert dates
df["target_doc"] = pd.to_datetime(
    df["target_doc"],
    errors="coerce"
)

df["revised_doc"] = pd.to_datetime(
    df["revised_doc"],
    errors="coerce"
)

# Create targets
df["cost_overrun"] = (
    df["revised_cost_cr"] > df["original_cost_cr"]
).astype(int)

df["time_overrun"] = (
    df["revised_doc"] > df["target_doc"]
).astype(int)

# Check targets
print(df["cost_overrun"].value_counts())
print(df["time_overrun"].value_counts())

Data format

In [13]:
print(df["target_doc"].head(10).to_list())
print(df["revised_doc"].head(10).to_list())

[Timestamp('2026-01-01 00:00:00'), Timestamp('2022-09-01 00:00:00'), Timestamp('2025-08-01 00:00:00'), Timestamp('2025-03-01 00:00:00'), Timestamp('2027-03-01 00:00:00'), Timestamp('2026-07-01 00:00:00'), Timestamp('2026-05-01 00:00:00'), Timestamp('2021-12-01 00:00:00'), Timestamp('2027-01-01 00:00:00'), Timestamp('2026-06-01 00:00:00')]
[Timestamp('2026-07-01 00:00:00'), Timestamp('2026-10-01 00:00:00'), Timestamp('2026-06-01 00:00:00'), Timestamp('2026-06-01 00:00:00'), NaT, Timestamp('2026-11-01 00:00:00'), Timestamp('2026-05-01 00:00:00'), Timestamp('2026-04-01 00:00:00'), NaT, Timestamp('2026-06-01 00:00:00')]


In [ ]:
Inspect Missing Dat

In [15]:
print("Missing target_doc:",
      df["target_doc"].isna().sum())

print("Missing revised_doc:",
      df["revised_doc"].isna().sum())

Missing target_doc: 0
Missing revised_doc: 354


In [14]:
print(df["target_doc"].dtype)
print(df["revised_doc"].dtype)

datetime64[ns]
datetime64[ns]


Missing Data

In [17]:
print("Missing target_doc:",
      df["target_doc"].isna().sum())

print("Missing revised_doc:",
      df["revised_doc"].isna().sum())

print(
    df[df["revised_doc"].isna()][
        ["project_code", "target_doc", "revised_doc"]
    ].head(20)
)

Missing target_doc: 0
Missing revised_doc: 354
    project_code target_doc revised_doc
4         612183 2027-03-01         NaT
8         619054 2027-01-01         NaT
26        615820 2031-07-01         NaT
27        615821 2030-11-01         NaT
28        400144 2029-03-01         NaT
29        400149 2029-03-01         NaT
30        400354 2029-03-01         NaT
32        613798 2026-03-01         NaT
33        613799 2026-03-01         NaT
36        615191 2029-03-01         NaT
37        616234 2033-03-01         NaT
38        617287 2030-03-01         NaT
39        617288 2029-03-01         NaT
40        619171 2032-03-01         NaT
41        619172 2029-03-01         NaT
42        400018 2029-03-01         NaT
43        400073 2032-03-01         NaT
44        400074 2029-03-01         NaT
45        400136 2032-03-01         NaT
46        400138 2032-03-01         NaT


In [19]:
# reporting month
print(df["reporting_month"].head(10).to_list())
print(df["reporting_month"].dtype)

print(df["reporting_month"].nunique())

['2026-04', '2026-04', '2026-04', '2026-04', '2026-04', '2026-04', '2026-04', '2026-04', '2026-04', '2026-04']
object
1


In [20]:
# Quick Check
print(
    df[
        [
            "approval_start_date",
            "revised_start_date",
            "target_doc",
            "revised_doc"
        ]
    ].head(10)
)

  approval_start_date revised_start_date target_doc revised_doc
0             03/2023            01/2024 2026-01-01  2026-07-01
1             06/2020            09/2020 2022-09-01  2026-10-01
2             12/2022            08/2023 2025-08-01  2026-06-01
3             12/2016            03/2018 2025-03-01  2026-06-01
4             08/2024            04/2025 2027-03-01         NaT
5             07/2024            07/2024 2026-07-01  2026-11-01
6             10/2018            10/2018 2026-05-01  2026-05-01
7             11/2017            12/2019 2021-12-01  2026-04-01
8             01/2024            07/2025 2027-01-01         NaT
9             06/2022            07/2022 2026-06-01  2026-06-01


In [21]:
print("Missing revised_doc:",
      df["revised_doc"].isna().sum())

Missing revised_doc: 354


Feature Engineering| Cost overrun

In [23]:
# Build the features Cost overrun

# Convert approval/revised start dates
df["approval_start"] = pd.to_datetime(
    df["approval_start_date"],
    format="%m/%Y",
    errors="coerce"
)

df["revised_start"] = pd.to_datetime(
    df["revised_start_date"],
    format="%m/%Y",
    errors="coerce"
)

# Start delay in months
df["start_delay_months"] = (
    (df["revised_start"].dt.year - df["approval_start"].dt.year) * 12
    + (df["revised_start"].dt.month - df["approval_start"].dt.month)
)

# Financial progress
df["financial_progress_pct"] = (
    df["cumulative_expenditure_cr"] /
    df["original_cost_cr"]
) * 100

# Difference between money spent and physical work completed
df["progress_gap"] = (
    df["financial_progress_pct"] -
    df["physical_progress_pct"]
)

In [24]:
features = [
    "ministry",
    "sector",
    "agency",
    "state",
    "original_cost_cr",
    "cumulative_expenditure_cr",
    "physical_progress_pct",
    "financial_progress_pct",
    "progress_gap",
    "start_delay_months"
]

X = df[features]

In [25]:
# Target y
y_cost = df["cost_overrun"]

In [26]:
# Split the data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_cost,
    test_size=0.2,
    random_state=42,
    stratify=y_cost
)

In [27]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, SimpleImputer

categorical_features = [
    "ministry",
    "sector",
    "agency",
    "state"
]

numeric_features = [
    "original_cost_cr",
    "cumulative_expenditure_cr",
    "physical_progress_pct",
    "financial_progress_pct",
    "progress_gap",
    "start_delay_months"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        )
    ]
)

In [28]:
# Train Logistic Regression
# Now we create a proper ML pipeline:

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

cost_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

In [29]:
# let's train
cost_model.fit(X_train, y_train)

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [30]:
# fix error check
print(X.isna().sum())

ministry                      0
sector                        0
agency                        0
state                         0
original_cost_cr              0
cumulative_expenditure_cr     0
physical_progress_pct         0
financial_progress_pct        0
progress_gap                  0
start_delay_months           11
dtype: int64


In [31]:
# fixed with Nan
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

categorical_features = [
    "ministry",
    "sector",
    "agency",
    "state"
]

numeric_features = [
    "original_cost_cr",
    "cumulative_expenditure_cr",
    "physical_progress_pct",
    "financial_progress_pct",
    "progress_gap",
    "start_delay_months"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

cost_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

In [32]:
# now train again
cost_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['original_cost_cr',
                                                   'cumulative_expenditure_cr',
                                                   'physical_progress_pct',
                                                   'financial_progress_pct',
                                                   'progress_gap',
                                                   'start_delay_months']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['ministry', 'sector',
                                                   'agency', 'state'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [33]:
# let's predict
y_pred = cost_model.predict(X_test)

y_prob = cost_model.predict_proba(X_test)[:, 1]

In [34]:
y_pred

array([0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0,
       0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1,
       1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0,
       0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0,
       1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1,
       0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0,
       0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1,

In [35]:
y_prob

array([0.36937314, 0.49377837, 0.52575053, 0.02486602, 0.1652818 ,
       0.07039249, 0.27811489, 0.08392131, 0.94461637, 0.05187198,
       0.06437171, 0.40236263, 0.83523331, 0.04458433, 0.28789109,
       0.12709095, 0.67482129, 0.50463696, 0.15884084, 0.51754385,
       0.65753227, 0.18970476, 0.07167155, 0.79561364, 0.28911299,
       0.91053688, 0.68969279, 0.0773643 , 0.44621556, 0.62332849,
       0.98701573, 0.45334202, 0.58903189, 0.21659719, 0.23645748,
       0.96306982, 0.78430471, 0.55697228, 0.34349155, 0.01819911,
       0.66069318, 0.61124779, 0.59360524, 0.9654362 , 0.70563437,
       0.34889635, 0.97064453, 0.0738575 , 0.07549132, 0.71523293,
       0.42389417, 0.00651169, 0.00972833, 0.89731521, 0.12863669,
       0.09449295, 0.07159446, 0.49860542, 0.06649601, 0.01471177,
       0.73399948, 0.27564568, 0.67154014, 0.21073047, 0.03128568,
       0.02261093, 0.87196437, 0.18456038, 0.31035916, 0.47029519,
       0.07431376, 0.05685857, 0.19285668, 0.59998001, 0.23523

In [36]:
# evaluate the Model
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC:")
print(roc_auc_score(y_test, y_prob))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.75      0.82       288
           1       0.54      0.77      0.63       109

    accuracy                           0.76       397
   macro avg       0.72      0.76      0.73       397
weighted avg       0.80      0.76      0.77       397

ROC-AUC:
0.8536888379204893
Confusion Matrix:
[[216  72]
 [ 25  84]]


In [37]:
# Save the Model
import joblib

joblib.dump(cost_model, "cost_overrun_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [38]:
# Download the model weight
from google.colab import files

files.download("cost_overrun_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>